# Movies Recommender System

I built two extremely minimalist predictive models to predict movie revenue and movie success and visualise which features influence the output (revenue and success respectively).

In this notebook, I will attempt at implementing a few recommendation algorithms (content based, popularity based and collaborative filtering) and try to build an ensemble of these models to come up with our final recommendation system. I had taken two MovieLens datasets.

* **The Full Dataset:** Consists of 26,000,000 ratings and 750,000 tag applications applied to 45,000 movies by 270,000 users. Includes tag genome data with 12 million relevance scores across 1,100 tags.
* **The Small Dataset:** Comprises of 100,000 ratings and 1,300 tag applications applied to 9,000 movies by 700 users.

I had build my Simple Recommender using movies from the *Full Dataset* whereas all personalised recommender systems will make use of the small dataset (due to the computing power I possess being very limited). As a first step, let us build our simple recommender system.

### 📦 Setup: Installing & Loading Libraries

- **`%pip install -q nltk scikit-surprise`** — Installs `nltk` (text processing) and `scikit-surprise` (collaborative filtering, used later). `-q` suppresses install logs.
- **`import importlib` / `import ssl`** — Built-in modules: `importlib` checks packages installed properly; `ssl` handles secure downloads.
- **`for module_name in ["nltk", "surprise"]: ...`** — Confirms both packages actually installed; stops early with a clear error if not.
- **`import nltk`** — Loads nltk into memory.
- **`try / except` block with `ssl`** — Workaround to disable strict SSL certificate checks, since it sometimes blocks `nltk.download()` (e.g. on corporate networks).
- **`nltk.download(...)` loop** — Downloads 4 NLTK data packages used later for text preprocessing:
  - `wordnet`, `omw-1.4` — word meaning/relationship database (used for stemming)
  - `punkt` — splits text into words/sentences
  - `stopwords` — common filler words to ignore ("the", "is", "and")

**Summary:** Pure setup — no modeling happens here.

In [1]:
%pip install -q nltk scikit-surprise

import importlib
import ssl

for module_name in ["nltk", "surprise"]:
    if importlib.util.find_spec(module_name) is None:
        raise ModuleNotFoundError(f"{module_name} is still unavailable after installation")

import nltk

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

for package in ["wordnet", "omw-1.4", "punkt", "stopwords"]:
    nltk.download(package, quiet=True)


Note: you may need to restart the kernel to use updated packages.


### 🔍 Sanity Check: Confirming the Python Environment

A quick diagnostic cell (not part of the actual recommender logic) to confirm which Python environment/interpreter is running.

- **`import sys`** — Loads Python's built-in `sys` module, used to inspect details about the interpreter itself.
- **`print(sys.executable)`** — Prints the file path of the Python interpreter currently in use. Useful for confirming packages are installing into the right environment (a common gotcha in Jupyter/Colab when multiple Python installs exist).
- **`print(sys.version)`** — Prints the exact Python version and build info.

**Why it's here:** Right after installing packages in the previous cell, this is a habit-check to make sure you're running the environment you think you are. It has no effect on the recommender system itself.

In [2]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\tirth\Desktop\MovieRS\myenv\Scripts\python.exe
3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


### 📚 Importing Core Libraries

Loads all tools needed throughout the notebook.

- `%matplotlib inline` — display plots inside the notebook
- `pandas`, `numpy` — data handling and numerical operations
- `matplotlib`, `seaborn` — visualization
- `scipy.stats` — statistical functions
- `literal_eval` — safely converts stringified lists/dicts (e.g. `"['Action','Drama']"`) into real Python objects
- `TfidfVectorizer`, `CountVectorizer` — convert text into numeric vectors for the content-based recommender
- `linear_kernel`, `cosine_similarity` — compute similarity between vectors
- `SnowballStemmer`, `WordNetLemmatizer`, `wordnet` — text preprocessing (reduce words to root form)
- `Reader`, `Dataset`, `SVD`, `cross_validate` — Surprise library tools for collaborative filtering
- `warnings.simplefilter('ignore')` — suppress warning messages for cleaner output

In [3]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from nltk.stem.snowball import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import wordnet
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate

import warnings; warnings.simplefilter('ignore')

## Simple Recommender

The Simple Recommender offers generalized recommnendations to every user based on movie popularity and (sometimes) genre. The basic idea behind this recommender is that movies that are more popular and more critically acclaimed will have a higher probability of being liked by the average audience. This model does not give personalized recommendations based on the user. 

The implementation of this model is extremely trivial. All we have to do is sort our movies based on ratings and popularity and display the top movies of our list. As an added step, we can pass in a genre argument to get the top movies of a particular genre. 

In [4]:
md = pd. read_csv('datasets/movies_metadata.csv')
md.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [5]:
md.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

### 🎭 Cleaning the "genres" Column

The `genres` column arrives as text that looks like a list of dictionaries, e.g. `"[{'id': 18, 'name': 'Drama'}]"`. This line converts it into a clean list of genre names, e.g. `['Drama', 'Comedy']`.

- **`.fillna('[]')`** — Replaces missing values with an empty-list string so nothing breaks downstream.
- **`.apply(literal_eval)`** — Converts the string into a real Python list of dictionaries.
- **`.apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])`** — Extracts just the `'name'` from each dictionary, giving a simple list of genre names. The `isinstance` check is a safety fallback in case parsing fails.

In [6]:
md['genres'] = md['genres'].fillna('[]').apply(literal_eval).apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

I use the TMDB Ratings to come up with our **Top Movies Chart.** I will use IMDB's *weighted rating* formula to construct my chart. Mathematically, it is represented as follows:

Weighted Rating (WR) = $(\frac{v}{v + m} . R) + (\frac{m}{v + m} . C)$

where,
* *v* is the number of votes for the movie
* *m* is the minimum votes required to be listed in the chart
* *R* is the average rating of the movie
* *C* is the mean vote across the whole report

The next step is to determine an appropriate value for *m*, the minimum votes required to be listed in the chart. We will use **95th percentile** as our cutoff. In other words, for a movie to feature in the charts, it must have more votes than at least 95% of the movies in the list.

I will build our overall Top 250 Chart and will define a function to build charts for a particular genre. Let's begin!

In [7]:
vote_counts = md[md['vote_count'].notnull()]['vote_count'].astype('int')
vote_averages = md[md['vote_average'].notnull()]['vote_average'].astype('int')
C = vote_averages.mean()
C

np.float64(5.244896612406511)

### ⭐ Calculating C (Mean Vote Across All Movies)

First step in building the IMDB Weighted Rating formula: `WR = (v/(v+m))·R + (m/(v+m))·C`

- **`vote_counts = ...`** — Filters out rows with missing `vote_count`, converts remaining values to integers.
- **`vote_averages = ...`** — Same, but for `vote_average`.
- **`C = vote_averages.mean()`** — The average rating across *all* movies. This acts as the "baseline" that pulls low-vote movies toward the overall average, so a movie with 2 votes of 10 can't outrank one with 10,000 votes averaging 8.
- **`C`** — Displays the computed value (~5.24).

### 🔢 Calculating m (Minimum Votes Cutoff)

The second key ingredient in the Weighted Rating formula — the minimum vote count a movie needs to qualify for the chart.

- **`m = vote_counts.quantile(0.95)`** — Finds the 95th percentile of vote counts. A movie must have more votes than 95% of all movies to qualify (comes out to ~434 votes).
- **`m`** — Displays the cutoff value.

**Why 95th percentile?** A strict threshold that filters out obscure, low-vote movies, ensuring the "Top Chart" only includes movies with meaningful audience feedback.

In [8]:
m = vote_counts.quantile(0.95)
m

np.float64(434.0)

In [9]:
md['year'] = pd.to_datetime(md['release_date'], errors='coerce').apply(lambda x: str(x).split('-')[0] if x != np.nan else np.nan)

### 📅 Extracting the Release Year

- **`md['year'] = pd.to_datetime(md['release_date'], errors='coerce').apply(lambda x: str(x).split('-')[0] if x != np.nan else np.nan)`** — Converts the `release_date` column to proper datetime objects (`errors='coerce'` turns any invalid/unparseable dates into `NaT` instead of crashing), then extracts just the year portion by converting each date to a string and splitting on `-` (since dates are formatted as `YYYY-MM-DD`, splitting on `-` and taking the first piece gives the year). Creates a new `year` column used later for filtering and display.

In [10]:
qualified = md[(md['vote_count'] >= m) & (md['vote_count'].notnull()) & (md['vote_average'].notnull())][['title', 'year', 'vote_count', 'vote_average', 'popularity', 'genres']]
qualified['vote_count'] = qualified['vote_count'].astype('int')
qualified['vote_average'] = qualified['vote_average'].astype('int')
qualified.shape

(2274, 6)

### ✅ Filtering Qualified Movies

- **`qualified = md[(md['vote_count'] >= m) & (md['vote_count'].notnull()) & (md['vote_average'].notnull())][['title', 'year', 'vote_count', 'vote_average', 'popularity', 'genres']]`** — Filters `md` to keep only movies with `vote_count` ≥ `m` (the 95th percentile cutoff) and with non-null vote data, then selects just the relevant columns needed for the chart.
- **`qualified['vote_count'] = qualified['vote_count'].astype('int')`** — Converts vote count to integer.
- **`qualified['vote_average'] = qualified['vote_average'].astype('int')`** — Converts vote average to integer.
- **`qualified.shape`** — Displays the (rows, columns) of the filtered DataFrame — confirms how many movies qualified (~2274, per the notebook).

Therefore, to qualify to be considered for the chart, a movie has to have at least **434 votes** on TMDB. We also see that the average rating for a movie on TMDB is **5.244** on a scale of 10. **2274** Movies qualify to be on our chart.

In [11]:
def weighted_rating(x):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m) * R) + (m/(m+v) * C)

In [12]:
qualified['wr'] = qualified.apply(weighted_rating, axis=1)

In [13]:
qualified = qualified.sort_values('wr', ascending=False).head(250)

### 🏆 Calculating Weighted Rating & Building the Top 250 Chart

- **`def weighted_rating(x): ...`** — Defines the IMDB Weighted Rating function: `v = x['vote_count']`, `R = x['vote_average']`, then returns `(v/(v+m) * R) + (m/(m+v) * C)`. This blends a movie's own rating (`R`) with the overall average (`C`), weighted by how many votes it has (`v`) relative to the cutoff (`m`) — movies with fewer votes get pulled closer to the overall average `C`, while movies with lots of votes rely more on their own rating `R`.
- **`qualified['wr'] = qualified.apply(weighted_rating, axis=1)`** — Applies that function to every row (`axis=1` means row-wise, not column-wise) and stores the result in a new `wr` (weighted rating) column.
- **`qualified = qualified.sort_values('wr', ascending=False).head(250)`** — Sorts all qualified movies by their weighted rating, highest first, and keeps only the top 250 — this becomes the final "Top Movies Chart."

### Top Movies

In [14]:
qualified.head(15)

,title,year,vote_count,vote_average,popularity,genres,wr
15480,Inception,2010,14075,8,29.108149,"[Action, Thriller, Science Fiction, Mystery, A...",7.917588
12481,The Dark Knight,2008,12269,8,123.167259,"[Drama, Action, Crime, Thriller]",7.905871
22879,Interstellar,2014,11187,8,32.213481,"[Adventure, Drama, Science Fiction]",7.897107
2843,Fight Club,1999,9678,8,63.869599,[Drama],7.881753
4863,The Lord of the Rings: The Fellowship of the Ring,2001,8892,8,32.070725,"[Adventure, Fantasy, Action]",7.871787
292,Pulp Fiction,1994,8670,8,140.950236,"[Thriller, Crime]",7.868660
314,The Shawshank Redemption,1994,8358,8,51.645403,"[Drama, Crime]",7.864000
7000,The Lord of the Rings: The Return of the King,2003,8226,8,29.324358,"[Adventure, Fantasy, Action]",7.861927
351,Forrest Gump,1994,8147,8,48.307194,"[Comedy, Drama, Romance]",7.860656
5814,The Lord of the Rings: The Two Towers,2002,7641,8,29.423537,"[Adventure, Fantasy, Action]",7.851924


We see that three Christopher Nolan Films, **Inception**, **The Dark Knight** and **Interstellar** occur at the very top of our chart. The chart also indicates a strong bias of TMDB Users towards particular genres and directors. 

Let us now construct our function that builds charts for particular genres. For this, we will use relax our default conditions to the **85th** percentile instead of 95. 

In [15]:
s = md.apply(lambda x: pd.Series(x['genres']),axis=1).stack().reset_index(level=1, drop=True)
s.name = 'genre'
gen_md = md.drop('genres', axis=1).join(s)

In [16]:
def build_chart(genre, percentile=0.85):
    df = gen_md[gen_md['genre'] == genre]
    vote_counts = df[df['vote_count'].notnull()]['vote_count'].astype('int')
    vote_averages = df[df['vote_average'].notnull()]['vote_average'].astype('int')
    C = vote_averages.mean()
    m = vote_counts.quantile(percentile)
    
    qualified = df[(df['vote_count'] >= m) & (df['vote_count'].notnull()) & (df['vote_average'].notnull())][['title', 'year', 'vote_count', 'vote_average', 'popularity']]
    qualified['vote_count'] = qualified['vote_count'].astype('int')
    qualified['vote_average'] = qualified['vote_average'].astype('int')
    
    qualified['wr'] = qualified.apply(lambda x: (x['vote_count']/(x['vote_count']+m) * x['vote_average']) + (m/(m+x['vote_count']) * C), axis=1)
    qualified = qualified.sort_values('wr', ascending=False).head(250)
    
    return qualified

### 🎭 Preparing Genre-Specific Charts

**Cell 15 — Exploding genres into separate rows:**
- **`s = md.apply(lambda x: pd.Series(x['genres']), axis=1).stack().reset_index(level=1, drop=True)`** — Each movie can have multiple genres (a list). This line "explodes" that list so each genre gets its own row (e.g., a movie with `['Drama', 'Comedy']` becomes two rows — one for Drama, one for Comedy). `pd.Series(x['genres'])` turns the list into a row of separate columns per movie, `.stack()` then stacks those columns into a single long column, and `.reset_index(level=1, drop=True)` cleans up the resulting index.
- **`s.name = 'genre'`** — Names this new Series `'genre'` so it can be joined back cleanly.
- **`gen_md = md.drop('genres', axis=1).join(s)`** — Drops the original `genres` list-column from `md`, then joins in the new single-genre-per-row `s` Series. Result: `gen_md` has one row per (movie, genre) pair instead of one row per movie.

**Cell 16 — The `build_chart` function:**
- **`def build_chart(genre, percentile=0.85): ...`** — A reusable function that repeats the same Weighted Rating logic as before (Cells 6–12), but filtered to a single genre, and using an **85th percentile** cutoff instead of 95th (a relaxed threshold since each genre has fewer movies to choose from than the whole dataset). It filters `gen_md` to the given genre, recalculates `C` and `m` just for that genre's movies, computes weighted ratings, sorts, and returns the top 250 for that genre.

Let us see our method in action by displaying the Top 15 Romance Movies (Romance almost didn't feature at all in our Generic Top Chart despite  being one of the most popular movie genres).

### Top Romance Movies

In [17]:
build_chart('Romance').head(15)

,title,year,vote_count,vote_average,popularity,wr
10309,Dilwale Dulhania Le Jayenge,1995,661,9,34.457024,8.565285
351,Forrest Gump,1994,8147,8,48.307194,7.971357
876,Vertigo,1958,1162,8,18.20822,7.811667
40251,Your Name.,2016,1030,8,34.461252,7.789489
883,Some Like It Hot,1959,835,8,11.845107,7.745154
1132,Cinema Paradiso,1988,834,8,14.177005,7.744878
19901,Paperman,2012,734,8,7.198633,7.713951
37863,Sing Street,2016,669,8,10.672862,7.689483
882,The Apartment,1960,498,8,11.994281,7.599317
38718,The Handmaiden,2016,453,8,16.727405,7.566166


The top romance movie according to our metrics is Bollywood's **Dilwale Dulhania Le Jayenge**. This Shahrukh Khan starrer also happens to be one of my personal favorites.

## Content Based Recommender

The recommender we built in the previous section suffers some severe limitations. For one, it gives the same recommendation to everyone, regardless of the user's personal taste. If a person who loves romantic movies (and hates action) were to look at our Top 15 Chart, s/he wouldn't probably like most of the movies. If s/he were to go one step further and look at our charts by genre, s/he wouldn't still be getting the best recommendations.

For instance, consider a person who loves *Dilwale Dulhania Le Jayenge*, *My Name is Khan* and *Kabhi Khushi Kabhi Gham*. One inference we can obtain is that the person loves the actor Shahrukh Khan and the director Karan Johar. Even if s/he were to access the romance chart, s/he wouldn't find these as the top recommendations.

To personalise our recommendations more, I am going to build an engine that computes similarity between movies based on certain metrics and suggests movies that are most similar to a particular movie that a user liked. Since we will be using movie metadata (or content) to build this engine, this also known as **Content Based Filtering.**

I will build two Content Based Recommenders based on:
* Movie Overviews and Taglines
* Movie Cast, Crew, Keywords and Genre

Also, as mentioned in the introduction, I will be using a subset of all the movies available to us due to limiting computing power available to me. 

In [18]:
links_small = pd.read_csv('datasets/links_small.csv')
links_small = links_small[links_small['tmdbId'].notnull()]['tmdbId'].astype('int')

In [19]:
md = md.drop([19730, 29503, 35587])

In [20]:
#Check EDA Notebook for how and why I got these indices.
md['id'] = md['id'].astype('int')

In [21]:
smd = md[md['id'].isin(links_small)]
smd.shape

(9099, 25)

### 🔗 Preparing the Small Movies Dataset

**Cell 18 — Loading movie ID links:**
- **`links_small = pd.read_csv('datasets/links_small.csv')`** — Loads a mapping file linking MovieLens IDs to TMDB IDs, for the smaller ~9,000-movie dataset used in all personalized recommenders (content-based, collaborative, hybrid) — used instead of the full 45,000-movie dataset purely due to limited compute power.
- **`links_small = links_small[links_small['tmdbId'].notnull()]['tmdbId'].astype('int')`** — Filters out rows with missing `tmdbId`, keeps just that column, and converts to integers — giving a clean list of valid TMDB IDs.

**Cell 19 — Dropping bad rows:**
- **`md = md.drop([19730, 29503, 35587])`** — Removes 3 specific rows (by index) from `md` that contain corrupted/malformed data (identified during EDA in the earlier notebook — these rows had bad `id` values that would break the next line).

**Cell 20 — Fixing the id column's type:**
- **`md['id'] = md['id'].astype('int')`** — Converts the `id` column from string/float to integer, so it can be properly matched against `links_small` (which is also integer type) in the next step.

**Cell 21 — Filtering to the small dataset:**
- **`smd = md[md['id'].isin(links_small)]`** — Creates `smd` ("small movies data") by keeping only rows in `md` whose `id` appears in `links_small`. This shrinks the dataset from ~45,000 movies down to ~9,099.
- **`smd.shape`** — Displays the resulting (rows, columns) to confirm the filtering worked.

We have **9099** movies avaiable in our small movies metadata dataset which is 5 times smaller than our original dataset of 45000 movies.

### Movie Description Based Recommender

Let us first try to build a recommender using movie descriptions and taglines. We do not have a quantitative metric to judge our machine's performance so this will have to be done qualitatively.

In [22]:
smd['tagline'] = smd['tagline'].fillna('')
smd['description'] = smd['overview'] + smd['tagline']
smd['description'] = smd['description'].fillna('')

In [23]:
tf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    min_df=1,
    stop_words='english'
)

tfidf_matrix = tf.fit_transform(smd['description'])

In [24]:
tfidf_matrix.shape

(9099, 268124)

### 📝 Building the Description-Based Content Recommender

**Cell 22 — Combining overview and tagline into one text field:**
- **`smd['tagline'] = smd['tagline'].fillna('')`** — Replaces missing `tagline` values with an empty string (so it doesn't break text concatenation next).
- **`smd['description'] = smd['overview'] + smd['tagline']`** — Combines each movie's `overview` (plot summary) and `tagline` into one text field called `description`.
- **`smd['description'] = smd['description'].fillna('')`** — Fills any remaining missing values in `description` with an empty string, as a final safety net (e.g. in case `overview` itself was missing).

**Cell 23 — Converting text into numeric vectors (TF-IDF):**
- **`tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1, stop_words='english')`** — Creates a TF-IDF (Term Frequency–Inverse Document Frequency) vectorizer, which converts text into numbers based on how important each word is to a movie's description relative to all other movies. Parameters:
  - `analyzer='word'` — work at the word level, not character level
  - `ngram_range=(1, 2)` — consider both single words *and* two-word phrases (e.g. "space" and "space odyssey")
  - `min_df=1` — include a word/phrase even if it appears in just 1 document
  - `stop_words='english'` — ignore common filler words like "the", "is", "a"
- **`tfidf_matrix = tf.fit_transform(smd['description'])`** — Learns the vocabulary from all movie descriptions and transforms each description into a numeric vector, producing a large sparse matrix (rows = movies, columns = words/phrases).

**Cell 24 — Checking the matrix size:**
- **`tfidf_matrix.shape`** — Displays (number of movies, number of unique words/phrases) to confirm the matrix was built correctly.

#### Cosine Similarity

I will be using the Cosine Similarity to calculate a numeric quantity that denotes the similarity between two movies. Mathematically, it is defined as follows:

$cosine(x,y) = \frac{x. y^\intercal}{||x||.||y||} $

Since we have used the TF-IDF Vectorizer, calculating the Dot Product will directly give us the Cosine Similarity Score. Therefore, we will use sklearn's **linear_kernel** instead of cosine_similarities since it is much faster.

In [25]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [26]:
cosine_sim[0]

array([1.        , 0.00680476, 0.        , ..., 0.        , 0.00344913,
       0.        ], shape=(9099,))

We now have a pairwise cosine similarity matrix for all the movies in our dataset. The next step is to write a function that returns the 30 most similar movies based on the cosine similarity score.

In [27]:
smd = smd.reset_index()
titles = smd['title']
indices = pd.Series(smd.index, index=smd['title'])

In [28]:
def get_recommendations(title):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:31]
    movie_indices = [i[0] for i in sim_scores]
    return titles.iloc[movie_indices]

### 🔍 Calculating Similarity & Building the Recommendation Function

**Cell 25 — Computing cosine similarity between all movies:**
- **`cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)`** — Computes pairwise similarity between every movie's TF-IDF vector and every other movie's vector, producing a square matrix (movies × movies) where each cell is a similarity score. `linear_kernel` is used instead of `cosine_similarity` because, since TF-IDF vectors are already normalized, the dot product (which is what `linear_kernel` computes) gives the same result as cosine similarity but is computed faster.

**Cell 26 — Peeking at one row of the similarity matrix:**
- **`cosine_sim[0]`** — Displays the similarity scores of the first movie against all other movies, just to sanity-check the matrix looks reasonable (values between 0 and 1, with a 1.0 for its similarity to itself).

**Cell 27 — Setting up lookup structures:**
- **`smd = smd.reset_index()`** — Resets the DataFrame's index to a clean 0, 1, 2... sequence (needed because rows were dropped/filtered earlier, so the old index had gaps that no longer match positions in `cosine_sim`).
- **`titles = smd['title']`** — Stores just the movie titles as a separate Series, for easy lookup later.
- **`indices = pd.Series(smd.index, index=smd['title'])`** — Creates a reverse-lookup Series: given a movie *title*, it returns that movie's *row position* — the opposite direction of normal indexing. This lets you look up a movie by name instead of by number.

**Cell 28 — Defining the recommendation function:**
- **`def get_recommendations(title): ...`** — Given a movie title, returns the 30 most similar movies:
  - `idx = indices[title]` — looks up the row position of the given movie
  - `sim_scores = list(enumerate(cosine_sim[idx]))` — pairs each other movie's index with its similarity score to this movie
  - `sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)` — sorts these pairs by similarity score, highest first
  - `sim_scores = sim_scores[1:31]` — skips the first result (which is always the movie itself, similarity = 1.0) and takes the next 30
  - `movie_indices = [i[0] for i in sim_scores]` — extracts just the row positions from those pairs
  - `return titles.iloc[movie_indices]` — looks up and returns the actual titles for those positions

We're all set. Let us now try and get the top recommendations for a few movies and see how good the recommendations are.

In [29]:
get_recommendations('The Godfather').head(10)

973      The Godfather: Part II
8387                 The Family
3509                       Made
4196         Johnny Dangerously
29               Shanghai Triad
5667                       Fury
2412             American Movie
1582    The Godfather: Part III
4221                    8 Women
2159              Summer of Sam
Name: title, dtype: str

In [30]:
get_recommendations('The Dark Knight').head(10)

7931                      The Dark Knight Rises
132                              Batman Forever
1113                             Batman Returns
8227    Batman: The Dark Knight Returns, Part 2
7565                 Batman: Under the Red Hood
524                                      Batman
7901                           Batman: Year One
2579               Batman: Mask of the Phantasm
2696                                        JFK
8165    Batman: The Dark Knight Returns, Part 1
Name: title, dtype: str

We see that for **The Dark Knight**, our system is able to identify it as a Batman film and subsequently recommend other Batman films as its top recommendations. But unfortunately, that is all this system can do at the moment. This is not of much use to most people as it doesn't take into considerations very important features such as cast, crew, director and genre, which determine the rating and the popularity of a movie. Someone who liked **The Dark Knight** probably likes it more because of Nolan and would hate **Batman Forever** and every other substandard movie in the Batman Franchise.

Therefore, we are going to use much more suggestive metadata than **Overview** and **Tagline**. In the next subsection, we will build a more sophisticated recommender that takes **genre**, **keywords**, **cast** and **crew** into consideration.

### Metadata Based Recommender

To build our standard metadata based content recommender, we will need to merge our current dataset with the crew and the keyword datasets. Let us prepare this data as our first step.

In [31]:
credits = pd.read_csv('datasets/credits.csv')
keywords = pd.read_csv('datasets/keywords.csv')

In [32]:
keywords['id'] = keywords['id'].astype('int')
credits['id'] = credits['id'].astype('int')
md['id'] = md['id'].astype('int')

In [33]:
md.shape

(45463, 25)

In [34]:
md = md.merge(credits, on='id')
md = md.merge(keywords, on='id')

In [35]:
smd = md[md['id'].isin(links_small)]
smd.shape

(9219, 28)

### 🎭 Merging in Cast, Crew & Keywords Data

**Cell 30 — Loading the extra datasets:**
- **`credits = pd.read_csv('datasets/credits.csv')`** — Loads cast and crew information (who acted in / worked on each movie).
- **`keywords = pd.read_csv('datasets/keywords.csv')`** — Loads plot keyword tags associated with each movie.

**Cell 31 — Matching data types for merging:**
- **`keywords['id'] = keywords['id'].astype('int')`**, **`credits['id'] = credits['id'].astype('int')`**, **`md['id'] = md['id'].astype('int')`** — Ensures the `id` column is the same integer type across all three DataFrames, since merging requires matching types on the join key.

**Cell 32 — Checking size before merging:**
- **`md.shape`** — Displays the current (rows, columns) of `md` as a checkpoint before merging.

**Cell 33 — Merging the datasets:**
- **`md = md.merge(credits, on='id')`** — Joins `credits` (cast/crew) into `md`, matching rows by movie `id`.
- **`md = md.merge(keywords, on='id')`** — Joins `keywords` into `md` the same way. Now `md` has genres, cast, crew, and keywords all in one DataFrame.

**Cell 34 — Re-filtering to the small dataset:**
- **`smd = md[md['id'].isin(links_small)]`** — Since `md` was rebuilt via merges, `smd` is recreated by filtering again to just the ~9,000 movies in the small dataset.
- **`smd.shape`** — Confirms the resulting size.

We now have our cast, crew, genres and credits, all in one dataframe. Let us wrangle this a little more using the following intuitions:

1. **Crew:** From the crew, we will only pick the director as our feature since the others don't contribute that much to the *feel* of the movie.
2. **Cast:** Choosing Cast is a little more tricky. Lesser known actors and minor roles do not really affect people's opinion of a movie. Therefore, we must only select the major characters and their respective actors. Arbitrarily we will choose the top 3 actors that appear in the credits list. 

In [36]:
smd['cast'] = smd['cast'].apply(literal_eval)
smd['crew'] = smd['crew'].apply(literal_eval)
smd['keywords'] = smd['keywords'].apply(literal_eval)
smd['cast_size'] = smd['cast'].apply(lambda x: len(x))
smd['crew_size'] = smd['crew'].apply(lambda x: len(x))

In [37]:
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [38]:
smd['director'] = smd['crew'].apply(get_director)

In [39]:
smd['cast'] = smd['cast'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])
smd['cast'] = smd['cast'].apply(lambda x: x[:3] if len(x) >=3 else x)

In [40]:
smd['keywords'] = smd['keywords'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

### 🎬 Parsing Cast, Crew & Keywords

**Cell 35 — Converting stringified data & measuring sizes:**
- **`smd['cast'] = smd['cast'].apply(literal_eval)`**, **`smd['crew'] = smd['crew'].apply(literal_eval)`**, **`smd['keywords'] = smd['keywords'].apply(literal_eval)`** — Like the `genres` column earlier, these columns arrive as text that looks like lists of dictionaries. `literal_eval` converts them into real Python lists.
- **`smd['cast_size'] = smd['cast'].apply(lambda x: len(x))`**, **`smd['crew_size'] = smd['crew'].apply(lambda x: len(x))`** — Creates new columns counting how many cast/crew members each movie has (informational, not used directly in the recommender logic yet).

**Cell 36 — Defining a function to extract the director:**
- **`def get_director(x): ...`** — Loops through a movie's crew list (list of dictionaries), and returns the `name` of the person whose `job` is `'Director'`. If no director is found, returns `np.nan` (missing value marker).

**Cell 37 — Applying it to create a director column:**
- **`smd['director'] = smd['crew'].apply(get_director)`** — Runs `get_director` on every movie's crew list, creating a new `director` column.

**Cell 38 — Simplifying and trimming the cast list:**
- **`smd['cast'] = smd['cast'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])`** — Extracts just actor names from the cast dictionaries (dropping other details like character name, order, etc.).
- **`smd['cast'] = smd['cast'].apply(lambda x: x[:3] if len(x) >=3 else x)`** — Keeps only the top 3 listed actors per movie (the reasoning: minor/lesser-known cast members don't meaningfully affect how similar two movies "feel" to a viewer).

**Cell 39 — Simplifying the keywords list:**
- **`smd['keywords'] = smd['keywords'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])`** — Same pattern as cast/genres: extracts just the keyword names from the list of dictionaries.

My approach to building the recommender is going to be extremely *hacky*. What I plan on doing is creating a metadata dump for every movie which consists of **genres, director, main actors and keywords.** I then use a **Count Vectorizer** to create our count matrix as we did in the Description Recommender. The remaining steps are similar to what we did earlier: we calculate the cosine similarities and return movies that are most similar.

These are steps I follow in the preparation of my genres and credits data:
1. **Strip Spaces and Convert to Lowercase** from all our features. This way, our engine will not confuse between **Johnny Depp** and **Johnny Galecki.** 
2. **Mention Director 2 times** to give it more weight relative to the entire cast.

In [41]:
smd['cast'] = smd['cast'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])

In [42]:
smd['director'] = (
    smd['director']
    .fillna('')
    .astype(str)
    .str.replace(' ', '', regex=False)
    .str.lower()
)
smd['director'] = smd['director'].apply(lambda x: [x,x])

### 🧹 Normalizing Cast Names & Director

**Cell 40 — Cleaning cast names:**
- **`smd['cast'] = smd['cast'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])`** — For each actor name, removes all spaces and converts to lowercase (e.g. "Johnny Depp" → "johnnydepp"). This prevents the model from later confusing two different people with similar first/last names (e.g. "Johnny Depp" vs "Johnny Galecki") when it treats each name as a single "token."

**Cell 41 — Cleaning and weighting the director:**
- **`smd['director'] = smd['director'].fillna('').astype(str).str.replace(' ', '', regex=False).str.lower()`** — Same cleanup as cast: fills missing directors with an empty string, removes spaces, and lowercases the name.
- **`smd['director'] = smd['director'].apply(lambda x: [x,x])`** — Duplicates the director's name into a 2-item list (e.g. `['christophernolan', 'christophernolan']`). This is a deliberate trick to give the director **more weight** than any single actor when everything gets combined into one text "soup" later — since the director appears twice, they count twice as much toward similarity.

#### Keywords

We will do a small amount of pre-processing of our keywords before putting them to any use. As a first step, we calculate the frequenct counts of every keyword that appears in the dataset.

In [43]:
s = smd.apply(lambda x: pd.Series(x['keywords']),axis=1).stack().reset_index(level=1, drop=True)
s.name = 'keyword'

In [44]:
s = s.value_counts()
s[:5]

keyword
independent film        610
woman director          550
murder                  399
duringcreditsstinger    327
based on novel          318
Name: count, dtype: int64

Keywords occur in frequencies ranging from 1 to 610. We do not have any use for keywords that occur only once. Therefore, these can be safely removed. Finally, we will convert every word to its stem so that words such as *Dogs* and *Dog* are considered the same.

In [45]:
s = s[s > 1]

In [46]:
stemmer = SnowballStemmer('english')
stemmer.stem('dogs')

'dog'

In [47]:
def filter_keywords(x):
    words = []
    for i in x:
        if i in s:
            words.append(i)
    return words

In [48]:
smd['keywords'] = smd['keywords'].apply(filter_keywords)
smd['keywords'] = smd['keywords'].apply(lambda x: [stemmer.stem(i) for i in x])
smd['keywords'] = smd['keywords'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])

In [49]:
smd['soup'] = smd['keywords'] + smd['cast'] + smd['director'] + smd['genres']
smd['soup'] = smd['soup'].apply(lambda x: ' '.join(x))

In [50]:
count = CountVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    min_df=1,
    stop_words='english'
)
count_matrix = count.fit_transform(smd['soup'])

In [51]:
cosine_sim = cosine_similarity(count_matrix, count_matrix)

In [52]:
smd = smd.reset_index()
titles = smd['title']
indices = pd.Series(smd.index, index=smd['title'])

### 🥣 Processing Keywords & Building the Metadata "Soup"

**Cell 42 — Counting keyword frequency (setup):**
- **`s = smd.apply(lambda x: pd.Series(x['keywords']), axis=1).stack().reset_index(level=1, drop=True)`** — Same "explode" trick used earlier for genres: turns each movie's list of keywords into one row per keyword, so frequency can be counted.
- **`s.name = 'keyword'`** — Names this new Series for clarity.

**Cell 43 — Counting how often each keyword appears:**
- **`s = s.value_counts()`** — Counts how many times each unique keyword appears across all movies.
- **`s[:5]`** — Displays the top 5 most frequent keywords, just to peek at the result.

**Cell 44 — Removing rare keywords:**
- **`s = s[s > 1]`** — Keeps only keywords that appear more than once across the dataset. Keywords used only once add noise without helping find similar movies, so they're dropped.

**Cell 45 — Setting up stemming:**
- **`stemmer = SnowballStemmer('english')`** — Creates a stemmer, a tool that reduces words to their root form (e.g. "dogs" → "dog", "running" → "run"), so different forms of the same word are treated as identical.
- **`stemmer.stem('dogs')`** — A quick test call to confirm the stemmer works as expected.

**Cell 46 — Defining a keyword filter function:**
- **`def filter_keywords(x): ...`** — Loops through a movie's keyword list and keeps only the keywords that survived the frequency filter (i.e., exist in `s` from Cell 44).

**Cell 47 — Applying the filter and stemming:**
- **`smd['keywords'] = smd['keywords'].apply(filter_keywords)`** — Removes rare keywords from each movie's keyword list.
- **`smd['keywords'] = smd['keywords'].apply(lambda x: [stemmer.stem(i) for i in x])`** — Stems each remaining keyword to its root form.
- **`smd['keywords'] = smd['keywords'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])`** — Same cleanup as cast/director: removes spaces and lowercases each keyword, so multi-word keywords become single tokens.

**Cell 48 — Building the "soup":**
- **`smd['soup'] = smd['keywords'] + smd['cast'] + smd['director'] + smd['genres']`** — Combines all four cleaned lists (keywords, cast, director, genres) into one big list per movie — this combined bag of tags is nicknamed the "soup."
- **`smd['soup'] = smd['soup'].apply(lambda x: ' '.join(x))`** — Joins that list into a single space-separated text string per movie (needed because the next step, CountVectorizer, expects text input, not lists).

**Cell 49 — Vectorizing the soup:**
- **`count = CountVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1, stop_words='english')`** — Similar to the earlier `TfidfVectorizer`, but simpler: `CountVectorizer` just counts how many times each word/phrase appears, without weighting by rarity. This is preferred here because the "soup" is a curated list of specific tags (names, keywords, genres) rather than natural prose — raw counts work well since we deliberately duplicated the director for weighting, and don't want TF-IDF's rarity-weighting to interfere with that.
- **`count_matrix = count.fit_transform(smd['soup'])`** — Learns the vocabulary from all movies' soups and converts each into a numeric count vector.

**Cell 50 — Recomputing similarity:**
- **`cosine_sim = cosine_similarity(count_matrix, count_matrix)`** — Computes pairwise similarity between every movie's soup vector, overwriting the earlier `cosine_sim` from the description-based recommender. (True `cosine_similarity` is used here, not `linear_kernel`, since count vectors — unlike TF-IDF vectors — aren't pre-normalized, so the dot-product shortcut doesn't apply.)

**Cell 51 — Rebuilding lookup structures:**
- **`smd = smd.reset_index()`**, **`titles = smd['title']`**, **`indices = pd.Series(smd.index, index=smd['title'])`** — Same pattern as Cell 26: resets the index and rebuilds the title-to-position lookup, since `smd` has changed since the description-based recommender was built.

We will reuse the get_recommendations function that we had written earlier. Since our cosine similarity scores have changed, we expect it to give us different (and probably better) results. Let us check for **The Dark Knight** again and see what recommendations I get this time around.

In [53]:
get_recommendations('The Dark Knight').head(10)

7991                 The Dark Knight Rises
6186                         Batman Begins
7619            Batman: Under the Red Hood
6587                          The Prestige
1122                        Batman Returns
8899               Kidnapping Mr. Heineken
5907                              Thursday
1252                        Batman & Robin
2077                             Following
9004    Batman v Superman: Dawn of Justice
Name: title, dtype: str

I am much more satisfied with the results I get this time around. The recommendations seem to have recognized other Christopher Nolan movies (due to the high weightage given to director) and put them as top recommendations. I enjoyed watching **The Dark Knight** as well as some of the other ones in the list including **Batman Begins**, **The Prestige** and **The Dark Knight Rises**. 

We can of course experiment on this engine by trying out different weights for our features (directors, actors, genres), limiting the number of keywords that can be used in the soup, weighing genres based on their frequency, only showing movies with the same languages, etc.

In [54]:
get_recommendations('Pulp Fiction').head(10)

1373         Jackie Brown
8877    The Hateful Eight
5172    Kill Bill: Vol. 2
4567                Basic
4736             S.W.A.T.
886        Reservoir Dogs
6903              Cleaner
4875    Kill Bill: Vol. 1
231         Kiss of Death
4282       The 51st State
Name: title, dtype: str

#### Popularity and Ratings

One thing that we notice about our recommendation system is that it recommends movies regardless of ratings and popularity. It is true that **Batman and Robin** has a lot of similar characters as compared to **The Dark Knight** but it was a terrible movie that shouldn't be recommended to anyone.

Therefore, we will add a mechanism to remove bad movies and return movies which are popular and have had a good critical response.

I will take the top 25 movies based on similarity scores and calculate the vote of the 60th percentile movie. Then, using this as the value of $m$, we will calculate the weighted rating of each movie using IMDB's formula like we did in the Simple Recommender section.

In [55]:
def improved_recommendations(title):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:26]
    movie_indices = [i[0] for i in sim_scores]
    
    movies = smd.iloc[movie_indices][['title', 'vote_count', 'vote_average', 'year']]
    vote_counts = movies[movies['vote_count'].notnull()]['vote_count'].astype('int')
    vote_averages = movies[movies['vote_average'].notnull()]['vote_average'].astype('int')
    C = vote_averages.mean()
    m = vote_counts.quantile(0.60)
    qualified = movies[(movies['vote_count'] >= m) & (movies['vote_count'].notnull()) & (movies['vote_average'].notnull())]
    qualified['vote_count'] = qualified['vote_count'].astype('int')
    qualified['vote_average'] = qualified['vote_average'].astype('int')
    qualified['wr'] = qualified.apply(weighted_rating, axis=1)
    qualified = qualified.sort_values('wr', ascending=False).head(10)
    return qualified

### ⭐ Improved Recommender: Filtering Out Bad Movies

The metadata-based recommender finds movies that are *similar* but ignores whether they're actually *good*. This function fixes that by re-applying the Weighted Rating logic on top of the similarity results.

- **`def improved_recommendations(title): ...`**
  - `idx = indices[title]` — looks up the movie's row position
  - `sim_scores = list(enumerate(cosine_sim[idx]))` — pairs every movie with its similarity score to this movie
  - `sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)` — sorts by similarity, highest first
  - `sim_scores = sim_scores[1:26]` — takes the top 25 similar movies (skipping the movie itself)
  - `movie_indices = [i[0] for i in sim_scores]` — extracts their row positions
  - `movies = smd.iloc[movie_indices][['title', 'vote_count', 'vote_average', 'year']]` — pulls out those 25 movies along with their vote data
  - `vote_counts`, `vote_averages`, `C = vote_averages.mean()` — same pattern as the Simple Recommender: computes the mean rating among just these 25 similar movies
  - `m = vote_counts.quantile(0.60)` — this time uses the **60th percentile** as the cutoff (much lower than the 95th used earlier, since we're only choosing among 25 already-similar movies, not the whole dataset)
  - `qualified = movies[...]` — filters to movies meeting that vote-count threshold
  - `qualified['wr'] = qualified.apply(weighted_rating, axis=1)` — reuses the same `weighted_rating` function defined way back in the Simple Recommender section
  - `qualified = qualified.sort_values('wr', ascending=False).head(10)` — sorts by weighted rating and returns the top 10

**Net effect:** Instead of just "similar movies," this returns "similar movies that are also well-rated," filtering out bad entries in the same franchise/style.

In [56]:
improved_recommendations('The Dark Knight')

,title,vote_count,vote_average,year,wr
6587,The Prestige,4510,8,2006,7.758148
7991,The Dark Knight Rises,9263,7,2012,6.921448
6186,Batman Begins,7511,7,2005,6.904127
7619,Batman: Under the Red Hood,459,7,2010,6.147016
2077,Following,363,7,1998,6.044272
1122,Batman Returns,1706,6,1992,5.846862
7517,Harry Brown,351,6,2009,5.582529
7986,Bullet to the Head,490,5,2013,5.115027
9004,Batman v Superman: Dawn of Justice,7189,5,2016,5.013943
1252,Batman & Robin,1447,4,1997,4.287233


In [57]:
improved_recommendations('Pulp Fiction')

,title,vote_count,vote_average,year,wr
886,Reservoir Dogs,3821,8,1992,7.718986
7240,Inglourious Basterds,6598,7,2009,6.891679
4875,Kill Bill: Vol. 1,5091,7,2003,6.862133
8877,The Hateful Eight,4405,7,2015,6.842588
5172,Kill Bill: Vol. 2,4061,7,2004,6.830542
1373,Jackie Brown,1580,7,1997,6.621790
8070,The Raid,1076,7,2011,6.495553
6752,Death Proof,1359,6,2007,5.817225
8518,Oldboy,632,5,2013,5.099705
4736,S.W.A.T.,780,5,2003,5.087550


Unfortunately, **Batman and Robin** does not disappear from our recommendation list. This is probably due to the fact that it is rated a 4, which is only slightly below average on TMDB. It certainly doesn't deserve a 4 when amazing movies like **The Dark Knight Rises** has only a 7. However, there is nothing much we can do about this. Therefore, we will conclude our Content Based Recommender section here and come back to it when we build a hybrid engine.

## Collaborative Filtering

Our content based engine suffers from some severe limitations. It is only capable of suggesting movies which are *close* to a certain movie. That is, it is not capable of capturing tastes and providing recommendations across genres.

Also, the engine that we built is not really personal in that it doesn't capture the personal tastes and biases of a user. Anyone querying our engine for recommendations based on a movie will receive the same recommendations for that movie, regardless of who s/he is.

Therefore, in this section, we will use a technique called **Collaborative Filtering** to make recommendations to Movie Watchers. Collaborative Filtering is based on the idea that users similar to a me can be used to predict how much I will like a particular product or service those users have used/experienced but I have not.

I will not be implementing Collaborative Filtering from scratch. Instead, I will use the **Surprise** library that used extremely powerful algorithms like **Singular Value Decomposition (SVD)** to minimise RMSE (Root Mean Square Error) and give great recommendations.

In [58]:
reader = Reader()

In [59]:
ratings = pd.read_csv('datasets/ratings_small.csv')
ratings.head()

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [60]:
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate

reader = Reader()
ratings = pd.read_csv('datasets/ratings_small.csv')

data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

svd = SVD()
cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

trainset = data.build_full_trainset()
svd.fit(trainset)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8966  0.8957  0.8971  0.8937  0.9023  0.8971  0.0029  
MAE (testset)     0.6910  0.6879  0.6913  0.6867  0.6957  0.6905  0.0031  
Fit time          2.18    2.29    2.27    2.50    2.13    2.27    0.13    
Test time         0.27    0.26    0.22    0.22    0.20    0.24    0.03    


### 🤝 Setting Up Collaborative Filtering

**Cell 55 — Creating a Reader:**
- **`reader = Reader()`** — Creates a `Reader` object from the Surprise library, which tells Surprise how to interpret the ratings data (e.g. rating scale). Default settings expect a rating scale of 1–5.

**Cell 56 — Loading the ratings dataset:**
- **`ratings = pd.read_csv('datasets/ratings_small.csv')`** — Loads user rating data: rows of `userId`, `movieId`, `rating` — i.e., what rating each user gave each movie.
- **`ratings.head()`** — Previews the first 5 rows.

**Cell 57 — Building the dataset and training an SVD model:**
- **`data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)`** — Converts the pandas DataFrame into a format Surprise understands, using only the 3 needed columns.
- **`svd = SVD()`** — Creates an SVD (Singular Value Decomposition) model — a matrix factorization algorithm that learns hidden patterns in user-movie ratings to predict how a user would rate a movie they haven't seen.
- **`cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)`** — Tests the model's accuracy using 5-fold cross-validation (splits data into 5 parts, trains/tests 5 times on different splits), measuring error using RMSE (Root Mean Square Error) and MAE (Mean Absolute Error). Lower is better for both.
- **`trainset = data.build_full_trainset()`** — Builds a training set using *all* the data (not just a cross-validation split), so the final model can learn from everything.
- **`svd.fit(trainset)`** — Trains the SVD model on the full dataset.

**Cell 58 — Re-running cross-validation (duplicate check):**
- **`cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)`** — Re-runs the same validation as above (likely just kept for displaying results clearly in its own cell/output). Notebook reports a mean RMSE of ~0.8963 — a strong result, meaning predicted ratings are typically less than 1 point off on a 1–5 scale.

In [61]:
from surprise.model_selection import cross_validate

cross_validate(
    svd,
    data,
    measures=['RMSE', 'MAE'],
    cv=5,
    verbose=True
)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8968  0.8927  0.9052  0.8947  0.8973  0.8973  0.0042  
MAE (testset)     0.6935  0.6883  0.6977  0.6871  0.6906  0.6914  0.0038  
Fit time          2.19    2.24    2.11    2.39    2.17    2.22    0.09    
Test time         0.25    0.46    0.23    0.33    0.20    0.29    0.09    


{'test_rmse': array([0.8967904 , 0.89271672, 0.90517975, 0.89474601, 0.89727754]),
 'test_mae': array([0.69350197, 0.68834552, 0.69768368, 0.68708605, 0.69058347]),
 'fit_time': (2.186476230621338,
  2.2383334636688232,
  2.1063995361328125,
  2.3879785537719727,
  2.174964189529419),
 'test_time': (0.2524750232696533,
  0.4596989154815674,
  0.225752592086792,
  0.32657694816589355,
  0.198897123336792)}

We get a mean **Root Mean Sqaure Error** of 0.8963 which is more than good enough for our case. Let us now train on our dataset and arrive at predictions.

In [62]:
trainset = data.build_full_trainset()
svd.fit(trainset)

Let us pick user 5000 and check the ratings s/he has given.

In [63]:
ratings[ratings['userId'] == 1]

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205
5,1,1263,2.0,1260759151
6,1,1287,2.0,1260759187
7,1,1293,2.0,1260759148
8,1,1339,3.5,1260759125
9,1,1343,2.0,1260759131


In [64]:
svd.predict(1, 302, 3)

Prediction(uid=1, iid=302, r_ui=3, est=np.float64(2.5860493651387473), details={'was_impossible': False})

For movie with ID 302, we get an estimated prediction of **2.686**. One startling feature of this recommender system is that it doesn't care what the movie is (or what it contains). It works purely on the basis of an assigned movie ID and tries to predict ratings based on how the other users have predicted the movie.

## Hybrid Recommender

![](https://www.toonpool.com/user/250/files/hybrid_20095.jpg)

In this section, I will try to build a simple hybrid recommender that brings together techniques we have implemented in the content based and collaborative filter based engines. This is how it will work:

* **Input:** User ID and the Title of a Movie
* **Output:** Similar movies sorted on the basis of expected ratings by that particular user.

In [65]:
def convert_int(x):
    try:
        return int(x)
    except:
        return np.nan

In [66]:
id_map = pd.read_csv('datasets/links_small.csv')[['movieId', 'tmdbId']]
id_map['tmdbId'] = id_map['tmdbId'].apply(convert_int)
id_map.columns = ['movieId', 'id']
id_map = id_map.merge(smd[['title', 'id']], on='id').set_index('title')
#id_map = id_map.set_index('tmdbId')

In [67]:
indices_map = id_map.set_index('id')

In [68]:
def hybrid(userId, title):
    idx = indices[title]
    tmdbId = id_map.loc[title]['id']
    #print(idx)
    movie_id = id_map.loc[title]['movieId']
    
    sim_scores = list(enumerate(cosine_sim[int(idx)]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:26]
    movie_indices = [i[0] for i in sim_scores]
    
    movies = smd.iloc[movie_indices][['title', 'vote_count', 'vote_average', 'year', 'id']]
    movies['est'] = movies['id'].apply(lambda x: svd.predict(userId, indices_map.loc[x]['movieId']).est)
    movies = movies.sort_values('est', ascending=False)
    return movies.head(10)

### 🔀 Building the Hybrid Recommender: Setup

**Cell — Helper function for safe conversion:**
- **`def convert_int(x): try: return int(x) except: return np.nan`** — A safe wrapper around `int()`. If a value can't be converted to an integer (e.g. it's missing or malformed), it returns `np.nan` instead of crashing the whole pipeline.

**Cell — Building an ID mapping table:**
- **`id_map = pd.read_csv('datasets/links_small.csv')[['movieId', 'tmdbId']]`** — Loads just the `movieId` (used by the ratings/Surprise data) and `tmdbId` (used by the content-based `smd`/`cosine_sim` data) columns. This is the bridge between the two systems, since collaborative filtering identifies movies by `movieId` while the content-based recommender identifies them by TMDB `id`.
- **`id_map['tmdbId'] = id_map['tmdbId'].apply(convert_int)`** — Safely converts `tmdbId` to integer using the helper function above.
- **`id_map.columns = ['movieId', 'id']`** — Renames `tmdbId` to `id` so it matches the column name used in `smd`.
- **`id_map = id_map.merge(smd[['title', 'id']], on='id').set_index('title')`** — Merges in movie titles (matching on `id`), then sets `title` as the index — so given a movie title, you can look up both its `movieId` and TMDB `id`.

**Cell — A second index for reverse lookup:**
- **`indices_map = id_map.set_index('id')`** — Creates another version of the same table, but indexed by TMDB `id` instead of title — needed later to go from a TMDB id to a `movieId` (which the SVD model understands).

In [69]:
hybrid(1, 'Avatar')

,title,vote_count,vote_average,year,id,est
522,Terminator 2: Judgment Day,4274.0,7.7,1991,280,3.327884
999,The Terminator,4208.0,7.4,1984,218,3.173230
962,Aliens,3282.0,7.7,1986,679,3.167181
2006,Fantastic Planet,140.0,7.6,1973,16306,3.035372
8622,X-Men: Days of Future Past,6155.0,7.5,2014,127585,3.001631
2826,Predator,2129.0,7.3,1987,106,2.964578
8833,Star Wars: The Force Awakens,7993.0,7.5,2015,140607,2.913826
7669,Alice in Wonderland,8.0,5.4,1933,25694,2.855991
1660,Return from Witch Mountain,38.0,5.6,1978,14822,2.782480
8357,Star Trek Into Darkness,4479.0,7.4,2013,54138,2.778374


In [70]:
hybrid(500, 'Avatar')

,title,vote_count,vote_average,year,id,est
7669,Alice in Wonderland,8.0,5.4,1933,25694,3.124431
1660,Return from Witch Mountain,38.0,5.6,1978,14822,3.106432
910,The Abyss,822.0,7.1,1989,2756,3.097745
8622,X-Men: Days of Future Past,6155.0,7.5,2014,127585,3.036735
3052,Sinbad and the Eye of the Tiger,39.0,6.3,1977,11940,3.034637
2006,Fantastic Planet,140.0,7.6,1973,16306,3.034071
7229,Dragonball Evolution,475.0,2.9,2009,14164,3.017165
2124,Superman II,642.0,6.5,1980,8536,3.003095
999,The Terminator,4208.0,7.4,1984,218,2.971906
962,Aliens,3282.0,7.7,1986,679,2.970980


We see that for our hybrid recommender, we get different recommendations for different users although the movie is the same. Hence, our recommendations are more personalized and tailored towards particular users.

## Conclusion

In this notebook, I have built 4 different recommendation engines based on different ideas and algorithms. They are as follows:

1. **Simple Recommender:** This system used overall TMDB Vote Count and Vote Averages to build Top Movies Charts, in general and for a specific genre. The IMDB Weighted Rating System was used to calculate ratings on which the sorting was finally performed.
2. **Content Based Recommender:** We built two content based engines; one that took movie overview and taglines as input and the other which took metadata such as cast, crew, genre and keywords to come up with predictions. We also deviced a simple filter to give greater preference to movies with more votes and higher ratings.
3. **Collaborative Filtering:** We used the powerful Surprise Library to build a collaborative filter based on single value decomposition. The RMSE obtained was less than 1 and the engine gave estimated ratings for a given user and movie.
4. **Hybrid Engine:** We brought together ideas from content and collaborative filterting to build an engine that gave movie suggestions to a particular user based on the estimated ratings that it had internally calculated for that user.